In [15]:
import os
import pandas as pd

# ==========================================
# KONFIGURASI
# ==========================================
DATA_DIR = '/Users/natasyays/Downloads/Documents/SEMPRO/bali dataset/Dataset/PerPantai'
OUTPUT_DIR = 'Standardized'
OUTPUT_FILE = os.path.join(OUTPUT_DIR, 'dataset_standardized.csv')
COVERAGE_FILE = os.path.join(OUTPUT_DIR, 'column_coverage_report.csv')

BEACHES = {
    'Kelingking':   'Kelingking Beach',
    'NusaDua':      'Nusa Dua Beach',
    'Sanur':        'Sanur Beach',
    'Kuta':         'Kuta Beach',
    'Pandawa':      'Pandawa Beach',
    'PadangPadang': 'Padang Padang Beach',
    'Seminyak':     'Seminyak Beach',
}
PLATFORMS = {'GR': 'GOOGLE', 'TA': 'TRIPADVISOR'}   # nilai 'source' sama dengan prepo lama
DEFAULT_BATCH = 'batch_1'

# ==========================================
# MAPPING KOLOM:  nama_standar : nama_kolom_asli
# ==========================================
TA_MAP = {
    'reviewer_name':     'user/name',
    'reviewer_location': 'user/userLocation/name',
    'trip_type':         'tripType',
    'travel_date':       'travelDate',          # YYYY-MM, hanya TA
    'published_date':    'publishedDate',
    'rating':            'rating',
    'text_original':     'text',
}

GR_MAP = {
    'reviewer_name':     'name',
    'published_date':    'publishedAtDate',
    'rating':            'stars',
    'text_original':     'text',
    'text_translated':   'textTranslated',       # dipakai untuk membentuk review_text, tidak disimpan
}

UNIFIED_COLUMNS = [
    'beach_name', 'source', 'scrape_batch',
    'review_id', 'reviewer_name', 'reviewer_location',
    'travel_date', 'trip_type', 'published_date',
    'rating',
    'review_text',      # teks untuk NLP: GR = textTranslated jika ada, jika kosong text asli; TA = text
    'text_original',    # teks asli -> ulasan hasil terjemahan = baris yang review_text != text_original
]


# ==========================================
# UTIL
# ==========================================
def read_csv_str(path):
    return pd.read_csv(path, dtype=str, keep_default_na=False, na_values=[''],
                       encoding='utf-8-sig')


def _blank_to_na(s):
    s = s.astype('string').str.strip()
    return s.mask(s == '')


def normalize_date(s):
    """ISO ('2026-08-30', '2026-09-21T04:55:12.833Z') atau M/D/YYYY ('9/10/2026') -> 'YYYY-MM-DD'.
    Tanggal Google (UTC) diambil tanggal UTC-nya."""
    s = _blank_to_na(s)
    iso = pd.to_datetime(s, errors='coerce', utc=True, format='ISO8601')
    mdy = pd.to_datetime(s, errors='coerce', format='%m/%d/%Y')
    out = iso.dt.strftime('%Y-%m-%d').fillna(mdy.dt.strftime('%Y-%m-%d'))
    n_mdy = int((iso.isna() & mdy.notna()).sum())
    n_fail = int((s.notna() & out.isna()).sum())
    return out.astype('string'), n_mdy, n_fail


def build_review_id(raw, platform):
    """Harus konsisten dengan kunci di 01_merge_scrape_batch.py."""
    if platform == 'GR':
        rid = _blank_to_na(raw['reviewId']) if 'reviewId' in raw else pd.Series(pd.NA, index=raw.index, dtype='string')
        return 'gr:' + rid
    url = _blank_to_na(raw['url']) if 'url' in raw else pd.Series(pd.NA, index=raw.index, dtype='string')
    num = url.str.extract(r'-r(\d+)-', expand=False).astype('string')
    if 'id' in raw:
        num = num.fillna(_blank_to_na(raw['id']))
    return 'ta:' + num


def standardize_one(path, beach_name, platform):
    raw = read_csv_str(path)
    raw = raw.loc[:, ~raw.columns.duplicated()]
    col_map = TA_MAP if platform == 'TA' else GR_MAP

    out = pd.DataFrame(index=raw.index)
    coverage = []
    for std_col, raw_col in col_map.items():
        exists = raw_col in raw.columns
        out[std_col] = _blank_to_na(raw[raw_col]) if exists else pd.NA
        coverage.append({'file': os.path.basename(path), 'std_column': std_col,
                         'raw_column': raw_col, 'exists': exists})

    out['beach_name'] = beach_name
    out['source'] = PLATFORMS[platform]
    out['scrape_batch'] = (_blank_to_na(raw['_scrape_batch']).fillna(DEFAULT_BATCH)
                           if '_scrape_batch' in raw.columns else DEFAULT_BATCH)
    out['review_id'] = build_review_id(raw, platform)

    # --- tanggal -> YYYY-MM-DD ---
    out['published_date'], n_mdy, n_fail = normalize_date(out['published_date'])
    warnings = []
    if n_mdy:
        warnings.append(f"{n_mdy:,} tanggal format M/D/YYYY (indikasi file pernah di-save Excel)")
    if n_fail:
        warnings.append(f"{n_fail:,} tanggal GAGAL diparse")

    # --- review_text: terjemahan jika ada, jika kosong teks asli ---
    if platform == 'GR':
        out['review_text'] = out['text_translated'].fillna(out['text_original'])
    else:
        out['review_text'] = out['text_original']

    out = out.reindex(columns=UNIFIED_COLUMNS)   # text_translated otomatis terbuang di sini
    return out, coverage, warnings


# ==========================================
# MAIN
# ==========================================
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    frames, coverage_all, missing_files = [], [], []

    for stem, beach_name in BEACHES.items():
        for platform in PLATFORMS:
            path = os.path.join(DATA_DIR, f'{stem}_{platform}.csv')
            if not os.path.exists(path):
                missing_files.append(os.path.basename(path))
                continue
            df, cov, warns = standardize_one(path, beach_name, platform)
            frames.append(df)
            coverage_all.extend(cov)
            n_no_id = df['review_id'].isna().sum()
            print(f"✅ {os.path.basename(path):<22} {len(df):>6,} baris"
                  f"{f'  ⚠️ {n_no_id} tanpa review_id' if n_no_id else ''}")
            for w in warns:
                print(f"     ⚠️ {w}")

    if missing_files:
        print(f"\n❌ FILE TIDAK DITEMUKAN ({len(missing_files)}): {missing_files}")
        print("   Output tetap dibuat, tapi dataset TIDAK LENGKAP.")

    if not frames:
        print("\n❌ TIDAK ADA FILE YANG TERBACA. Proses dihentikan.")
        print(f"   DATA_DIR  : {DATA_DIR}")
        print(f"   Ada?      : {os.path.exists(DATA_DIR)}")
        if os.path.exists(DATA_DIR):
            csvs = sorted(f for f in os.listdir(DATA_DIR) if f.lower().endswith('.csv'))
            print(f"   CSV di sana ({len(csvs)}): {csvs[:20]}")
            print("   -> Nama file harus berpola {Pantai}_GR.csv / {Pantai}_TA.csv, mis. NusaDua_TA.csv")
        else:
            print("   -> Folder tidak ada. Perbaiki DATA_DIR (cek typo, atau file sudah dipindah).")
        return

    all_df = pd.concat(frames, ignore_index=True)

    # Cek ID ganda lintas file (mis. file yang sama terdaftar 2x)
    dup = all_df['review_id'].notna() & all_df.duplicated(subset=['review_id'], keep=False)
    if dup.any():
        print(f"\n⚠️ {dup.sum():,} baris punya review_id ganda lintas file -> cek, lalu dedup di prepo.")

    all_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')

    cov_df = pd.DataFrame(coverage_all)
    cov_df.to_csv(COVERAGE_FILE, index=False)

    # Laporan hanya muncul jika ada kolom yang BENAR-BENAR TIDAK ADA di file
    missing = cov_df[~cov_df['exists']]
    if len(missing):
        print("\n" + "=" * 70)
        print("⚠️ KOLOM TIDAK ADA DI FILE (akan kosong di hasil)")
        print("=" * 70)
        print(missing[['file', 'raw_column']].to_string(index=False))

    print("\n" + "=" * 70)
    print("JUMLAH ULASAN PER PANTAI x PLATFORM x BATCH")
    print("=" * 70)
    print(pd.crosstab([all_df['beach_name'], all_df['source']], all_df['scrape_batch'],
                      margins=True, margins_name='TOTAL').to_string())

    print(f"\n💾 Tersimpan: {OUTPUT_FILE} ({len(all_df):,} baris)")
    print(f"💾 Laporan kolom: {COVERAGE_FILE}")


if __name__ == '__main__':
    main()

✅ Kelingking_GR.csv      10,000 baris
✅ Kelingking_TA.csv         997 baris
✅ NusaDua_GR.csv          1,608 baris
✅ NusaDua_TA.csv            999 baris
✅ Sanur_GR.csv            5,000 baris
✅ Sanur_TA.csv              999 baris
✅ Kuta_GR.csv            10,000 baris
✅ Kuta_TA.csv               999 baris
✅ Pandawa_GR.csv         10,000 baris
✅ Pandawa_TA.csv            999 baris
✅ PadangPadang_GR.csv     9,149 baris
✅ PadangPadang_TA.csv       999 baris
✅ Seminyak_GR.csv         5,200 baris
✅ Seminyak_TA.csv           999 baris


/var/folders/sp/r0xpyvgj56z_q0cs84tz3b700000gn/T/ipykernel_65570/868947607.py:167: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_df = pd.concat(frames, ignore_index=True)



JUMLAH ULASAN PER PANTAI x PLATFORM x BATCH
scrape_batch                     batch_1  TOTAL
beach_name          source                     
Kelingking Beach    GOOGLE         10000  10000
                    TRIPADVISOR      997    997
Kuta Beach          GOOGLE         10000  10000
                    TRIPADVISOR      999    999
Nusa Dua Beach      GOOGLE          1608   1608
                    TRIPADVISOR      999    999
Padang Padang Beach GOOGLE          9149   9149
                    TRIPADVISOR      999    999
Pandawa Beach       GOOGLE         10000  10000
                    TRIPADVISOR      999    999
Sanur Beach         GOOGLE          5000   5000
                    TRIPADVISOR      999    999
Seminyak Beach      GOOGLE          5200   5200
                    TRIPADVISOR      999    999
TOTAL                              57948  57948

💾 Tersimpan: Standardized/dataset_standardized.csv (57,948 baris)
💾 Laporan kolom: Standardized/column_coverage_report.csv
